## Exercises on Model-Based Reinforcement Learning

These paper-and-pencil exercises reinforce Chapter 06: learning a model (transition counts and reward means) from experience, planning with the learned model, the Dyna-Q architecture (direct RL + planning), expected vs. sampled model backups, and trajectory sampling. Notation follows the course convention: $\hat p(s'\mid s,a)$ and $\hat r(s,a,s')$ are the *learned* model, $Q(s,a)$ the action-value estimate, the reward for acting at $t$ is $R_{t+1}$.

### Exercise 6.1 — Learning a model from experience

For a fixed state–action pair $(s,a)$ the agent has observed the following three transitions (in order):

$\displaystyle (s,a)\to s_1,\ r=1; \qquad (s,a)\to s_1,\ r=1; \qquad (s,a)\to s_2,\ r=0.$

Using maximum-likelihood (count-based) estimation:

1. Build the transition model $\hat p(s'\mid s,a)$.
2. Build the reward model $\hat r(s,a,s')$ (the mean reward per transition), using the incremental-mean update.
3. Compute the expected reward $\hat r(s,a)$.

**Step 1 — Transition model from counts.** With $N(s,a,s')$ the number of times $(s,a)$ led to $s'$ and $N(s,a)=\sum_{s'}N(s,a,s')$:

$\displaystyle N(s,a)=3,\quad N(s,a,s_1)=2,\quad N(s,a,s_2)=1,$
$\displaystyle \hat p(s_1\mid s,a)=\frac{2}{3},\qquad \hat p(s_2\mid s,a)=\frac{1}{3}.$

**Step 2 — Reward model (incremental mean).** The chapter stores the mean reward per $(s,a,s')$ triple, updated by $\bar r \leftarrow \bar r + \tfrac{1}{N}(r-\bar r)$. For $s_1$ the two observed rewards are both $1$:

$\displaystyle \bar r_1 = 0 \xrightarrow{r=1,\,N=1} 0+\tfrac11(1-0)=1 \xrightarrow{r=1,\,N=2} 1+\tfrac12(1-1)=1 \;\Rightarrow\; \hat r(s,a,s_1)=1.$

For $s_2$ a single reward $0$: $\ \hat r(s,a,s_2)=0.$

**Step 3 — Expected reward.** Marginalise over next states with the learned model:

$\displaystyle \hat r(s,a)=\sum_{s'}\hat p(s'\mid s,a)\,\hat r(s,a,s') = \tfrac23(1)+\tfrac13(0)=\tfrac23.$

**Key concept**

A model-based agent *estimates the MDP from data*: transition probabilities become normalised visit counts, and rewards become running means. These estimates are exactly what planning will treat "as if" they were the true dynamics.

### Exercise 6.2 — One real step plus planning (Dyna-Q)

A deterministic chain has states $A \to B \to G$ ($G$ terminal, $Q(G,\cdot)=0$). The agent has already learned the model: $(A,\text{right})\to B,\ r=0$ and $(B,\text{right})\to G,\ r=1$. All $Q$-values start at $0$; use $\gamma=1,\ \alpha=0.5$.

1. The agent takes a **real** step $(B,\text{right})\to G$. Perform the direct Q-learning update.
2. Then perform **two planning updates** using experiences sampled from the model: first the pair $(A,\text{right})$, then $(B,\text{right})$.
3. Comment on what planning achieved.

**Step 1 — Direct RL update** from the real transition $(B,\text{right})\to G$, reward $1$:

$\displaystyle Q(B,\text{right}) \leftarrow 0 + 0.5\big[\,1 + \gamma\max_a Q(G,a) - 0\,\big] = 0 + 0.5(1) = 0.5.$

**Step 2 — Planning update 1**, sampled model experience $(A,\text{right})\to B,\ r=0$:

$\displaystyle Q(A,\text{right}) \leftarrow 0 + 0.5\big[\,0 + \gamma\max_a Q(B,a) - 0\,\big] = 0.5\,(1\cdot 0.5) = 0.25.$

**Step 3 — Planning update 2**, sampled model experience $(B,\text{right})\to G,\ r=1$:

$\displaystyle Q(B,\text{right}) \leftarrow 0.5 + 0.5\big[\,1 + 0 - 0.5\,\big] = 0.5 + 0.25 = 0.75.$

**Step 4 — Interpretation.** After a *single* real interaction, planning has already assigned value to $Q(A,\text{right})=0.25$ — even though the agent never physically stood in $A$ during this update — and has further sharpened $Q(B,\text{right})$. Planning replays the learned model to propagate value that pure model-free learning would only obtain through additional real experience.

**Key concept**

Dyna-Q interleaves **direct RL** (learn from real transitions) with **planning** (learn from model-simulated transitions). Both use the *same* Q-learning update; planning simply manufactures extra, cheap experience, which is decisive when real interaction is expensive.

### Exercise 6.3 — Why planning propagates credit faster

Consider a *deterministic* chain of $N$ non-terminal states $s_1\to s_2\to\cdots\to s_N\to G$, where only the final transition into $G$ yields reward $+1$; $Q$ is initialised to $0$, $\gamma=1$, and the agent acts greedily/optimally along the chain each episode (online, in-place updates).

1. Using **one-step Q-learning only** (no planning), how many complete episodes are needed, at minimum, before $Q(s_1,\cdot)$ becomes non-zero?
2. How can **Dyna-Q** achieve the same propagation within a single episode?

**Step 1 — One-step Q-learning propagation.** Within one episode the states are updated in the order visited, $s_1, s_2, \dots, s_N$. When $s_i$ is updated, its target uses $Q(s_{i+1},\cdot)$ **as it currently stands**. On the first episode only $Q(s_N,\cdot)$ becomes non-zero (its successor is the rewarding terminal); every earlier $Q(s_i,\cdot)$ still bootstraps from a zero successor. So the non-zero "frontier" moves back exactly **one state per episode**. To reach $s_1$ therefore requires

$\displaystyle N-1 \text{ episodes (at minimum).}$

**Step 2 — Dyna-Q.** After the single real episode (which makes $Q(s_N,\cdot)$ non-zero), the agent can run **planning updates in reverse** using the learned deterministic model — updating $s_{N-1}$ from the now-updated $s_N$, then $s_{N-2}$, and so on — propagating the reward all the way to $s_1$ **within the same episode**, using no additional real steps.

**Step 3 — Interpretation.** One-step model-free backups move information one link per episode; planning can chain many simulated backups per real step, so the number of *real* interactions needed collapses from $O(N)$ episodes to $O(1)$.

**Key concept**

The bottleneck in model-free learning is the *rate of credit assignment*, not the amount of computation. A model lets the agent "think" (replay simulated transitions) between actions, trading cheap computation for expensive real experience — the central promise of model-based RL.

### Exercise 6.4 — Expected (full) model backup

From the learned **stochastic** model of a pair $(s,a)$:

$\displaystyle \hat p(s_1\mid s,a)=0.7,\quad \hat p(s_2\mid s,a)=0.3,\qquad \hat r(s,a,s_1)=1,\quad \hat r(s,a,s_2)=0,$

with $\gamma=0.9$ and current greedy values $\max_{a'}Q(s_1,a')=10$, $\max_{a'}Q(s_2,a')=0$. Compute the **expected** (full) one-step backup

$\displaystyle Q(s,a) \leftarrow \sum_{s'}\hat p(s'\mid s,a)\big[\,\hat r(s,a,s') + \gamma\max_{a'}Q(s',a')\,\big],$

and contrast it with a single **sampled** Dyna-Q backup.

**Step 1 — Expected backup.** Average over both possible next states, weighted by the learned probabilities:

$\displaystyle Q(s,a) = 0.7\big[\,1 + 0.9(10)\,\big] + 0.3\big[\,0 + 0.9(0)\,\big] = 0.7(10) + 0.3(0) = 7.$

**Step 2 — Sampled backup (Dyna-Q style).** A single planning step draws *one* next state from $\hat p$:

- with prob. $0.7$ it draws $s_1$, giving target $1+0.9(10)=10$;
- with prob. $0.3$ it draws $s_2$, giving target $0+0.9(0)=0$.

The sampled target is $10$ or $0$; **in expectation** it equals the full backup $7$, but any single sample is noisy.

**Step 3 — Interpretation.** The expected backup (a "full" dynamic-programming update on the learned model) is exact but costs a sum over all next states; the sampled backup is cheap ($O(1)$ per update) but high-variance. Dyna-Q uses *many* cheap sampled backups; expected backups suit small branching factors.

**Key concept**

Planning can back up **in expectation** (sum over the model's next-state distribution, like value iteration) or **by sampling** (one simulated transition at a time, like Dyna-Q). The two agree in the mean; the choice trades computational cost per update against variance.

### Exercise 6.5 — Trajectory sampling vs. uniform sampling

An MDP has four relevant states. Under the current greedy policy, the **on-policy visitation frequencies** are proportional to $1:2:3:4$ (state $s_4$, adjacent to the goal, is visited most). Two planning strategies distribute their simulated updates differently:

- **Uniform (Dyna-Q):** pick the state to update uniformly at random among the four visited states.
- **Trajectory sampling:** pick states in proportion to the on-policy visitation frequencies.

Over $100$ planning updates, compute the expected number of updates each strategy devotes to $s_4$, and comment on the consequence.

**Step 1 — Normalise the on-policy distribution.** $\ d_\pi = \dfrac{(1,2,3,4)}{1+2+3+4} = (0.1, 0.2, 0.3, 0.4).$ So $d_\pi(s_4)=0.4$.

**Step 2 — Expected updates to $s_4$ over 100 planning steps.**

$\displaystyle \text{Uniform: } 100\times\tfrac14 = 25, \qquad \text{Trajectory sampling: } 100\times 0.4 = 40.$

**Step 3 — Interpretation.** Trajectory sampling concentrates $40\%$ of its effort on the goal-adjacent state $s_4$ (vs. $25\%$ for uniform), and correspondingly *less* on rarely visited states. Because value must be learned where the agent actually goes, this focus usually yields faster improvement of the current policy — at the cost of neglecting parts of the state space the policy currently avoids (which uniform sampling keeps covering).

**Key concept**

*Where* you spend planning updates matters. Trajectory (on-policy) sampling directs computation to the states the agent is likely to encounter, mirroring the real experience distribution; uniform sampling spreads effort evenly, exploring more of the model but wasting updates on irrelevant states.